# Evaluating Multiple LM Outputs (External)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# imports
import json
import os
import pandas as pd
import importlib.util
import sys
from os.path import join
from copy import deepcopy
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.additions.eval.extraction import \
    format_features, format_model_info
from stat_genie.blade_pipeline.additions.analysis.conclusion import \
    write_final_answer_code, make_conclusion
from stat_genie.blade_pipeline.additions.analysis.fix_code import \
    check_and_fix_code

In [3]:
# load files
analysis_subdir_path_1 = "analysis1_output"
analysis_subdir_path_2 = "analysis2_output"
analysis_subdir_path_3 = "analysis3_output"

multirun_filename_1 = "multirun_analyses.json"
multirun_filename_2 = "multirun_analyses.json"
multirun_filename_3 = "multirun_analyses.json"

# use both files to get analysis code paths
multirun_path_1 = join(analysis_subdir_path_1, multirun_filename_1)
multirun_path_2 = join(analysis_subdir_path_2, multirun_filename_2)
multirun_path_3 = join(analysis_subdir_path_3, multirun_filename_3)

with open(multirun_path_1, "r") as file:
    multirun_analyses_1 = json.load(file)

with open(multirun_path_2, "r") as file:
    multirun_analyses_2 = json.load(file)
    
with open(multirun_path_3, "r") as file:
    multirun_analyses_3 = json.load(file)

num_analyses_1 = multirun_analyses_1['n']
num_analyses_2 = multirun_analyses_2['n']
num_analyses_3 = multirun_analyses_3['n']

analysis_code_filenames_1 = [f"llm_analysis_{i}.py" for i in range(num_analyses_1)]
analysis_code_filenames_2 = [f"llm_analysis_{i}.py" for i in range(num_analyses_2)]
analysis_code_filenames_3 = [f"llm_analysis_{i}.py" for i in range(num_analyses_3)]

analysis_code_paths_1 = [join(analysis_subdir_path_1, filename)
                         for filename in analysis_code_filenames_1]

analysis_code_paths_2 = [join(analysis_subdir_path_2, filename)
                         for filename in analysis_code_filenames_2]

analysis_code_paths_3 = [join(analysis_subdir_path_3, filename)
                         for filename in analysis_code_filenames_3]

In [4]:
llm_provider = "openai"
llm_model = "gpt-5-mini"
llm_assistant = llm(provider=llm_provider, model=llm_model)

[2025-12-02 10:39:52.85][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [5]:
features_1 = format_features(multirun_analyses_1, num_analyses_1, llm_assistant)
features_2 = format_features(multirun_analyses_2, num_analyses_2, llm_assistant)
features_3 = format_features(multirun_analyses_3, num_analyses_3, llm_assistant)

[2025-12-02 10:39:54.16][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 10:40:10.87][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  16.71 seconds
[2025-12-02 10:40:10.88][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-02 10:40:10.90][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 10:40:18.71][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  7.81 seconds
[2025-12-02 10:40:18.73][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-02 10:40:18.78][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 10:40:34.46][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  15.6

In [6]:
model_info_1 = format_model_info(multirun_analyses_1, num_analyses_1, llm_assistant)
model_info_2 = format_model_info(multirun_analyses_2, num_analyses_2, llm_assistant)
model_info_3 = format_model_info(multirun_analyses_3, num_analyses_3, llm_assistant)

[2025-12-02 10:55:41.84][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 10:55:55.74][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  13.90 seconds
[2025-12-02 10:55:55.74][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-02 10:55:55.76][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 10:56:07.04][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  11.27 seconds
[2025-12-02 10:56:07.04][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-02 10:56:07.07][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 10:56:18.08][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  11.

[2025-12-02 10:56:40.75][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  9.69 seconds
[2025-12-02 10:56:40.76][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-02 10:56:40.78][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 10:57:08.04][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  27.26 seconds
[2025-12-02 10:57:08.04][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-02 10:57:08.08][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 10:57:20.72][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  12.64 seconds
[2025-12-02 10:57:20.73][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


In [7]:
# load dataset, need more user-friendly input method later
dataset_name = multirun_analyses_1['dataset_name']
dataset_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                    "datasets", dataset_name, "data.csv")
absolute_dataset_path = os.path.abspath(dataset_path)
data = pd.read_csv(dataset_path)

In [18]:
# check that code works
analysis_code_paths = [analysis_code_paths_1, analysis_code_paths_2,
                       analysis_code_paths_3]
for i in range(len(analysis_code_paths)):
    for j, analysis_code_path in enumerate(analysis_code_paths[i]):
        # get absolute path using relative path so that the helper function works correctly
        absolute_path = os.path.abspath(analysis_code_path)
        # call helper function to ensure code correctness
        num_iterations = check_and_fix_code(f"llm_analysis_{j}",
                                            absolute_path,
                                            "transform_model",
                                            llm_provider,
                                            llm_model,
                                            dataset_path=absolute_dataset_path,
                                            verbose=True)
        print(f"Analysis {i} iteration {j} required {num_iterations} correction iterations.")

/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


Negative Binomial model failed: Cannot interpret 'Int64Dtype()' as a data type
OLS model for log_ndam15 failed: Cannot interpret 'Int64Dtype()' as a data type
Analysis 0 iteration 0 required 0 correction iterations.


/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:2014: RuntimeWarning: divide by zero encountered in divide
  self.het_scale = (self.wresid / (1 - h))**2
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


Analysis 0 iteration 1 required 0 correction iterations.


/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


Analysis 0 iteration 2 required 0 correction iterations.
Error during runtime:
  File "/accounts/grad/zachrewolinski/research/stat-genie/src/stat_genie/blade_pipeline/additions/analysis/fix_code.py", line 93, in is_code_correct
    model_results = model_func(transformed_df)
  File "/accounts/grad/zachrewolinski/research/stat-genie/examples/separating_pipeline/analysis2_output/llm_analysis_0.py", line 97, in model
    model_cont = smf.ols(formula_cont, data=df).fit(cov_type='HC3')
  File "/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/base/model.py", line 203, in from_formula
    tmp = handle_formula_data(data, None, formula, depth=eval_env,
  File "/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/formula/formulatools.py", line 63, in handle_formula_data
    result = dmatrices(formula, Y, depth, return_type='dataframe',
  File "/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/pyt

/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 9, but rank is 0
  warnings.warn('covariance of constraints does not have full '
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/base/model.py:1923: RuntimeWarning: invalid value encountered in divide
  F /= J
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10

                            OLS Regression Results                            
Dep. Variable:             log_deaths   R-squared:                         nan
Model:                            OLS   Adj. R-squared:                    nan
Method:                 Least Squares   F-statistic:                       nan
Date:                Tue, 02 Dec 2025   Prob (F-statistic):                nan
Time:                        11:17:36   Log-Likelihood:                    inf
No. Observations:                  94   AIC:                              -inf
Df Residuals:                      87   BIC:                              -inf
Df Model:                           6                                         
Covariance Type:                  HC3                                         
                                                                       coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------

/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


Analysis 2 iteration 1 required 0 correction iterations.
Error during runtime:
  File "/accounts/grad/zachrewolinski/research/stat-genie/src/stat_genie/blade_pipeline/additions/analysis/fix_code.py", line 93, in is_code_correct
    model_results = model_func(transformed_df)
  File "/accounts/grad/zachrewolinski/research/stat-genie/examples/separating_pipeline/analysis3_output/llm_analysis_2.py", line 138, in model
    model = sm.OLS(y, X).fit(cov_type='HC3')
  File "/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/regression/linear_model.py", line 921, in __init__
    super().__init__(endog, exog, missing=missing,
  File "/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/regression/linear_model.py", line 746, in __init__
    super().__init__(endog, exog, missing=missing,
  File "/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/regression/linear_

In [19]:
transform_functions_1 = {}
transform_functions_2 = {}
transform_functions_3 = {}
model_functions_1 = {}
model_functions_2 = {}
model_functions_3 = {}

# ----- get the transform and model functions for the first input group -----
for i, analysis_code_path in enumerate(analysis_code_paths_1):
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_1_{i}"] = module
    spec.loader.exec_module(module)
    
    transform_functions_1[i] = module.transform
    model_functions_1[i] = module.model

# ----- get the transform and model functions for the second input group -----
for i, analysis_code_path in enumerate(analysis_code_paths_2):
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_2_{i}"] = module
    spec.loader.exec_module(module)

    transform_functions_2[i] = module.transform
    model_functions_2[i] = module.model

# ----- get the transform and model functions for the third input group -----
for i, analysis_code_path in enumerate(analysis_code_paths_3):
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_3_{i}"] = module
    spec.loader.exec_module(module)

    transform_functions_3[i] = module.transform
    model_functions_3[i] = module.model

[autoreload of _llm_analysis_0_0 failed: Traceback (most recent call last):
  File "/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
  File "/accounts/grad/zachrewolinski/.pyenv/versions/3.10.18/lib/python3.10/importlib/__init__.py", line 168, in reload
    raise ModuleNotFoundError(f"spec not found for the module {name!r}", name=name)
ModuleNotFoundError: spec not found for the module '_llm_analysis_0_0'
]


                            OLS Regression Results                            
Dep. Variable:             log_deaths   R-squared:                         nan
Model:                            OLS   Adj. R-squared:                    nan
Method:                 Least Squares   F-statistic:                       nan
Date:                Tue, 02 Dec 2025   Prob (F-statistic):                nan
Time:                        11:24:06   Log-Likelihood:                    inf
No. Observations:                  94   AIC:                              -inf
Df Residuals:                      87   BIC:                              -inf
Df Model:                           6                                         
Covariance Type:                  HC3                                         
                                                                       coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------

[autoreload of _llm_analysis_2_0 failed: Traceback (most recent call last):
  File "/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
  File "/accounts/grad/zachrewolinski/.pyenv/versions/3.10.18/lib/python3.10/importlib/__init__.py", line 168, in reload
    raise ModuleNotFoundError(f"spec not found for the module {name!r}", name=name)
ModuleNotFoundError: spec not found for the module '_llm_analysis_2_0'
]
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])
/accounts/grad/zachrewolinski/research/sta

In [20]:
transformed_datasets_1 = {}
for i, transform_func in transform_functions_1.items():
    try:
        transformed_datasets_1[i] = transform_func(data.copy())
        print(f"[Transform 1-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Transform 1-{i}] Failed with error: {e}")
        transformed_datasets_1[i] = None

transformed_datasets_2 = {}
for i, transform_func in transform_functions_2.items():
    try:
        transformed_datasets_2[i] = transform_func(data.copy())
        print(f"[Transform 2-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Transform 2-{i}] Failed with error: {e}")
        transformed_datasets_2[i] = None
        
transformed_datasets_3 = {}
for i, transform_func in transform_functions_3.items():
    try:
        transformed_datasets_3[i] = transform_func(data.copy())
        print(f"[Transform 3-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Transform 3-{i}] Failed with error: {e}")
        transformed_datasets_3[i] = None


model_results_1 = {}
for i, model_func in model_functions_1.items():
    try:
        if transformed_datasets_1[i] is None:
            print(f"[Model 1-{i}] Skipping — transform failed.")
            continue

        model_results_1[i] = model_func(transformed_datasets_1[i].copy())
        print(f"[Model 1-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Model 1-{i}] Failed with error: {e}")
        model_results_1[i] = None

model_results_2 = {}
for i, model_func in model_functions_2.items():
    try:
        if transformed_datasets_2[i] is None:
            print(f"[Model 2-{i}] Skipping — transform failed.")
            continue

        model_results_2[i] = model_func(transformed_datasets_2[i].copy())
        print(f"[Model 2-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Model 2-{i}] Failed with error: {e}")
        model_results_2[i] = None
        
model_results_3 = {}
for i, model_func in model_functions_3.items():
    try:
        if transformed_datasets_3[i] is None:
            print(f"[Model 3-{i}] Skipping — transform failed.")
            continue

        model_results_3[i] = model_func(transformed_datasets_3[i].copy())
        print(f"[Model 3-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Model 3-{i}] Failed with error: {e}")
        model_results_3[i] = None

[Transform 1-0] Completed successfully.
[Transform 1-1] Completed successfully.
[Transform 1-2] Completed successfully.
[Transform 2-0] Completed successfully.
[Transform 2-1] Completed successfully.
[Transform 2-2] Completed successfully.
[Transform 3-0] Completed successfully.
[Transform 3-1] Completed successfully.
[Transform 3-2] Completed successfully.
Negative Binomial model failed: Cannot interpret 'Int64Dtype()' as a data type
OLS model for log_ndam15 failed: Cannot interpret 'Int64Dtype()' as a data type
[Model 1-0] Completed successfully.
[Model 1-1] Completed successfully.


/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:2014: RuntimeWarning: divide by zero encountered in divide
  self.het_scale = (self.wresid / (1 - h))**2
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion paramet

[Model 1-2] Completed successfully.
[Model 2-0] Completed successfully.
                            OLS Regression Results                            
Dep. Variable:             log_deaths   R-squared:                         nan
Model:                            OLS   Adj. R-squared:                    nan
Method:                 Least Squares   F-statistic:                       nan
Date:                Tue, 02 Dec 2025   Prob (F-statistic):                nan
Time:                        11:24:07   Log-Likelihood:                    inf
No. Observations:                  94   AIC:                              -inf
Df Residuals:                      87   BIC:                              -inf
Df Model:                           6                                         
Covariance Type:                  HC3                                         
                                                                       coef    std err          z      P>|z|      [0.025      0.975]
-----

/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 9, but rank is 0
  warnings.warn('covariance of constraints does not have full '
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/base/model.py:1923: RuntimeWarning: invalid value encountered in divide
  F /= J
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10

[Model 3-1] Completed successfully.
[Model 3-2] Completed successfully.


In [21]:
info_json_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                      "datasets", dataset_name, "info.json")
with open(info_json_path, "r") as file:
    info_json = json.load(file)

task = info_json['research_questions']

for i in range(num_analyses_1):

    independent_variable = features_1[i]['independent_variables']
    dependent_variable = features_1[i]['response_variables']

    model_code = multirun_analyses_1['analyses'][str(i)]['m_code']
    model_output = model_results_1[i]

    write_final_answer_code(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        os.path.abspath(analysis_subdir_path_1),
        i,
        model_output
    )

for i in range(num_analyses_2):

    independent_variable = features_2[i]['independent_variables']
    dependent_variable = features_2[i]['response_variables']

    model_code = multirun_analyses_2['analyses'][str(i)]['m_code']
    model_output = model_results_2[i]

    write_final_answer_code(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        os.path.abspath(analysis_subdir_path_2),
        i,
        model_output
    )

for i in range(num_analyses_3):

    independent_variable = features_3[i]['independent_variables']
    dependent_variable = features_3[i]['response_variables']

    model_code = multirun_analyses_3['analyses'][str(i)]['m_code']
    model_output = model_results_3[i]

    write_final_answer_code(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        os.path.abspath(analysis_subdir_path_3),
        i,
        model_output
    )


[2025-12-02 11:24:10.99][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 11:24:59.97][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  48.98 seconds
[2025-12-02 11:24:59.98][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-02 11:25:00.01][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 11:25:39.75][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  39.73 seconds
[2025-12-02 11:25:39.75][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-02 11:25:39.79][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 11:26:13.87][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  34.

In [22]:
answer_code_paths_1 = [join(analysis_subdir_path_1, f"llm_answer_{i}.py") for i in range(num_analyses_1)]
answer_code_paths_2 = [join(analysis_subdir_path_2, f"llm_answer_{i}.py") for i in range(num_analyses_2)]
answer_code_paths_3 = [join(analysis_subdir_path_3, f"llm_answer_{i}.py") for i in range(num_analyses_3)]

In [23]:
# check that code works
answer_code_paths = [answer_code_paths_1, answer_code_paths_2,
                     answer_code_paths_3]
model_results = [model_results_1, model_results_2,
                 model_results_3]
for i in range(len(answer_code_paths)):
    for j, answer_code_path in enumerate(answer_code_paths[i]):
        # get absolute path using relative path so that the helper function works correctly
        absolute_path = os.path.abspath(answer_code_path)
        # call helper function to ensure code correctness
        num_iterations = check_and_fix_code(f"llm_answer_{j}",
                                            absolute_path,
                                            "final_answer",
                                            llm_provider,
                                            llm_model,
                                            model_output=model_results[i][j],
                                            verbose=False)
        print(f"Answer {i} iteration {j} required {num_iterations} correction iterations.")

Answer 0 iteration 0 required 0 correction iterations.
Answer 0 iteration 1 required 0 correction iterations.
Answer 0 iteration 2 required 0 correction iterations.
Answer 1 iteration 0 required 0 correction iterations.
Answer 1 iteration 1 required 0 correction iterations.
Answer 1 iteration 2 required 0 correction iterations.
[2025-12-02 11:29:30.75][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-02 11:29:31.03][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 11:29:58.58][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  27.55 seconds
[2025-12-02 11:29:58.58][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
Answer 2 iteration 0 required 1 correction iterations.
Answer 2 iteration 1 required 0 correction iteratio

In [24]:
# get final answer functions in dict
final_answer_functions_1 = {}
final_answer_functions_2 = {}
final_answer_functions_3 = {}

# ----- get the final answer functions for the first input group -----
for i, answer_code_path in enumerate(answer_code_paths_1):
    spec = importlib.util.spec_from_file_location(f"llm_answer_{i}",
                                                  answer_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_answer_1_{i}"] = module
    spec.loader.exec_module(module)
    
    final_answer_functions_1[i] = module.extract_final_answer

# ----- get the final answer functions for the second input group -----
for i, answer_code_path in enumerate(answer_code_paths_2):
    spec = importlib.util.spec_from_file_location(f"llm_answer_{i}",
                                                  answer_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_answer_2_{i}"] = module
    spec.loader.exec_module(module)

    final_answer_functions_2[i] = module.extract_final_answer
    
# ----- get the final answer functions for the third input group -----
for i, answer_code_path in enumerate(answer_code_paths_3):
    spec = importlib.util.spec_from_file_location(f"llm_answer_{i}",
                                                  answer_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_answer_3_{i}"] = module
    spec.loader.exec_module(module)

    final_answer_functions_3[i] = module.extract_final_answer


In [25]:
final_answers_1 = {}
for i, final_answer_func in final_answer_functions_1.items():
    try:
        model_output = deepcopy(model_results_1[i])
        final_answers_1[i] = final_answer_func(model_output)
        print(f"[Answer 1-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Answer 1-{i}] Failed with error: {e}")
        final_answers_1[i] = None

final_answers_2 = {}
for i, final_answer_func in final_answer_functions_2.items():
    try:
        model_output = deepcopy(model_results_2[i])
        final_answers_2[i] = final_answer_func(model_output)
        print(f"[Answer 2-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Answer 2-{i}] Failed with error: {e}")
        final_answers_2[i] = None
        
final_answers_3 = {}
for i, final_answer_func in final_answer_functions_3.items():
    try:
        model_output = deepcopy(model_results_3[i])
        final_answers_3[i] = final_answer_func(model_output)
        print(f"[Answer 3-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Answer 3-{i}] Failed with error: {e}")
        final_answers_3[i] = None

[Answer 1-0] Completed successfully.
[Answer 1-1] Completed successfully.
[Answer 1-2] Completed successfully.
[Answer 2-0] Completed successfully.
[Answer 2-1] Completed successfully.
[Answer 2-2] Completed successfully.
[Answer 3-0] Completed successfully.
[Answer 3-1] Completed successfully.
[Answer 3-2] Completed successfully.


In [26]:
conclusions_1 = {}

for i in range(num_analyses_1):
    independent_variable = features_1[i]['independent_variables']
    dependent_variable = features_1[i]['response_variables']

    model_code = multirun_analyses_1['analyses'][str(i)]['m_code']
    # read in final answer code from file "llm_answer_{i}.py" at analysis_subdir_path_1
    code_filename = f"llm_answer_{i}.py"
    code_path = os.path.abspath(os.path.join(analysis_subdir_path_1, code_filename))
    with open(code_path, "r", encoding="utf-8") as f:
        interpretation_code_str = f.read()
    interpretation_output = final_answers_1[i]

    conclusions_1[i] = make_conclusion(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        interpretation_code_str,
        interpretation_output
    )


conclusions_2 = {}

for i in range(num_analyses_2):
    independent_variable = features_2[i]['independent_variables']
    dependent_variable = features_2[i]['response_variables']

    model_code = multirun_analyses_2['analyses'][str(i)]['m_code']

    # read in final answer code from file "llm_answer_{i}.py" at analysis_subdir_path_2
    code_filename = f"llm_answer_{i}.py"
    code_path = os.path.abspath(os.path.join(analysis_subdir_path_2, code_filename))
    with open(code_path, "r", encoding="utf-8") as f:
        interpretation_code_str = f.read()
    interpretation_output = final_answers_2[i] if i < len(final_answers_2) else None

    conclusions_2[i] = make_conclusion(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        interpretation_code_str,
        interpretation_output
    )
    
conclusions_3 = {}

for i in range(num_analyses_3):
    independent_variable = features_3[i]['independent_variables']
    dependent_variable = features_3[i]['response_variables']

    model_code = multirun_analyses_3['analyses'][str(i)]['m_code']

    # read in final answer code from file "llm_answer_{i}.py" at analysis_subdir_path_3
    code_filename = f"llm_answer_{i}.py"
    code_path = os.path.abspath(os.path.join(analysis_subdir_path_3, code_filename))
    with open(code_path, "r", encoding="utf-8") as f:
        interpretation_code_str = f.read()
    interpretation_output = final_answers_3[i] if i < len(final_answers_3) else None

    conclusions_3[i] = make_conclusion(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        interpretation_code_str,
        interpretation_output
    )

[2025-12-02 11:30:43.89][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 11:30:48.40][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  4.52 seconds
[2025-12-02 11:30:48.40][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-02 11:30:48.43][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 11:30:56.19][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  7.77 seconds
[2025-12-02 11:30:56.20][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-02 11:30:56.21][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 11:31:01.29][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  5.08 

In [27]:
llm_judge = llm(provider=llm_provider, model=llm_model)
data_head = data.head(10)

[2025-12-02 11:31:38.14][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [28]:
judge_system_prompt = (
    "You are a meticulous research design evaluator. "
    "Your role is to compare two experimental trials methodologically **and interpretively**.\n\n"
    "You will go through the following reasoning plan step-by-step (internally):\n"
    "1. Understand the research question and dataset context.\n"
    "2. Examine independent, control, and response variables for both trials.\n"
    "3. Analyze the model specifications for structural or methodological similarity.\n"
    "4. Focus more on the content, less on the format.\n"
    "5. Assess whether the trials' conclusions are logically consistent given their setups.\n"
    "6. Detect whether either input is None, invalid, erroneous, or incomplete.\n"
    "   - If **one trial** shows errors or missing components but the other is valid, "
    "     impose a **strong penalty** (reduce all category scores by at least 1 point, "
    "     and cap overall similarity at 2).\n"
    "7. Synthesize your evaluation across all components.\n"
    "8. Output a numerical rating for each category.\n\n"
    "DO NOT include your reasoning — only the final dictionary.\n\n"
    "Scoring scale:\n"
    "1 = completely different\n"
    "2 = somewhat different\n"
    "3 = moderately similar\n"
    "4 = very similar\n"
    "5 = almost identical\n\n"
    "Return output **strictly in dictionary format**:\n"
    "{\n"
    "  \"independent_variables\": <number>,\n"
    "  \"control_variables\": <number>,\n"
    "  \"response_variables\": <number>,\n"
    "  \"model_specification\": <number>,\n"
    "  \"conclusions\": <number>,\n"
    "  \"overall_similarity\": <number>\n"
    "}"
)


def make_judge_prompt(task, data_head, featA, featB, modelA, modelB, conclA, conclB):
    return (
        f"Research Question / Context:\n{task}\n\n"
        "Here is a sample of the dataset to understand the structure and variables:\n"
        f"{data_head}\n\n"
        "Compare the two trials methodologically and interpretively based on the provided variables, model specifications, and conclusions.\n\n"
        "==================== TRIAL A ====================\n\n"
        "Independent Variables:\n"
        f"{featA['independent_variables']}\n\n"
        "Control Variables:\n"
        f"{featA.get('control_variables')}\n\n"
        "Response Variables:\n"
        f"{featA['response_variables']}\n\n"
        "Model Specification:\n"
        f"{modelA}\n\n"
        "Conclusion:\n"
        f"{conclA}\n\n"
        "==================== TRIAL B ====================\n\n"
        "Independent Variables:\n"
        f"{featB['independent_variables']}\n\n"
        "Control Variables:\n"
        f"{featB.get('control_variables')}\n\n"
        "Response Variables:\n"
        f"{featB['response_variables']}\n\n"
        "Model Specification:\n"
        f"{modelB}\n\n"
        "Conclusion:\n"
        f"{conclB}\n\n"
        "Now, following your reasoning plan, provide similarity ratings as JSON only."
    )


In [29]:
### judge results within each group and between each group.
# within each group there should be 3 choose 2 = 3 pairwise comparisons
# between each group there should be 3 x 3 = 9 pairwise comparisons

In [30]:
### begin with within-group performance
within_group = {1: {}, 2: {}, 3: {}}
for i in range(num_analyses_1):
    for j in range(i + 1, num_analyses_1):
        prompt = make_judge_prompt(
            task,
            data_head,
            features_1[i],
            features_1[j],
            multirun_analyses_1['analyses'][str(i)]['m_code'],
            multirun_analyses_1['analyses'][str(j)]['m_code'],
            conclusions_1[i],
            conclusions_1[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        within_group[1][(i, j)] = response_dict
for i in range(num_analyses_2):
    for j in range(i + 1, num_analyses_2):
        prompt = make_judge_prompt(
            task,
            data_head,
            features_2[i],
            features_2[j],
            multirun_analyses_2['analyses'][str(i)]['m_code'],
            multirun_analyses_2['analyses'][str(j)]['m_code'],
            conclusions_2[i],
            conclusions_2[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        within_group[2][(i, j)] = response_dict
for i in range(num_analyses_3):
    for j in range(i + 1, num_analyses_3):
        prompt = make_judge_prompt(
            task,
            data_head,
            features_3[i],
            features_3[j],
            multirun_analyses_3['analyses'][str(i)]['m_code'],
            multirun_analyses_3['analyses'][str(j)]['m_code'],
            conclusions_3[i],
            conclusions_3[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        within_group[3][(i, j)] = response_dict

[2025-12-02 11:31:39.71][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 11:31:58.76][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  19.05 seconds
[2025-12-02 11:31:58.76][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-02 11:31:58.80][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 11:32:13.59][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  14.79 seconds
[2025-12-02 11:32:13.61][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-02 11:32:13.65][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 11:32:23.95][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  10.

In [31]:
### now do between-group performance
between_group = { (1, 2): {}, (1, 3): {}, (2, 3): {} }
for i in range(num_analyses_1):
    for j in range(num_analyses_2):
        prompt = make_judge_prompt(
            task,
            data_head,
            features_1[i],
            features_2[j],
            multirun_analyses_1['analyses'][str(i)]['m_code'],
            multirun_analyses_2['analyses'][str(j)]['m_code'],
            conclusions_1[i],
            conclusions_2[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        between_group[(1, 2)][(i, j)] = response_dict
for i in range(num_analyses_1):
    for j in range(num_analyses_3):
        prompt = make_judge_prompt(
            task,
            data_head,
            features_1[i],
            features_3[j],
            multirun_analyses_1['analyses'][str(i)]['m_code'],
            multirun_analyses_3['analyses'][str(j)]['m_code'],
            conclusions_1[i],
            conclusions_3[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        between_group[(1, 3)][(i, j)] = response_dict
for i in range(num_analyses_2):
    for j in range(num_analyses_3):
        prompt = make_judge_prompt(
            task,
            data_head,
            features_2[i],
            features_3[j],
            multirun_analyses_2['analyses'][str(i)]['m_code'],
            multirun_analyses_3['analyses'][str(j)]['m_code'],
            conclusions_2[i],
            conclusions_3[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        between_group[(2, 3)][(i, j)] = response_dict

[2025-12-02 11:34:00.74][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 11:34:15.39][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  14.65 seconds
[2025-12-02 11:34:15.40][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-02 11:34:15.44][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 11:34:26.29][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  10.84 seconds
[2025-12-02 11:34:26.29][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-02 11:34:26.36][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-02 11:34:36.67][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  10.

In [32]:
within_group

{1: {(0, 1): {'independent_variables': 5,
   'control_variables': 5,
   'response_variables': 4,
   'model_specification': 5,
   'conclusions': 2,
   'overall_similarity': 4},
  (0, 2): {'independent_variables': 3,
   'control_variables': 3,
   'response_variables': 3,
   'model_specification': 3,
   'conclusions': 1,
   'overall_similarity': 2},
  (1, 2): {'independent_variables': 5,
   'control_variables': 3,
   'response_variables': 5,
   'model_specification': 4,
   'conclusions': 4,
   'overall_similarity': 4}},
 2: {(0, 1): {'independent_variables': 5,
   'control_variables': 4,
   'response_variables': 5,
   'model_specification': 4,
   'conclusions': 5,
   'overall_similarity': 4},
  (0, 2): {'independent_variables': 4,
   'control_variables': 4,
   'response_variables': 4,
   'model_specification': 3,
   'conclusions': 4,
   'overall_similarity': 2},
  (1, 2): {'independent_variables': 4,
   'control_variables': 4,
   'response_variables': 5,
   'model_specification': 3,
   'c

In [33]:
between_group

{(1,
  2): {(0, 0): {'independent_variables': 4,
   'control_variables': 4,
   'response_variables': 2,
   'model_specification': 3,
   'conclusions': 4,
   'overall_similarity': 2}, (0, 1): {'independent_variables': 5,
   'control_variables': 4,
   'response_variables': 4,
   'model_specification': 3,
   'conclusions': 5,
   'overall_similarity': 4}, (0, 2): {'independent_variables': 5,
   'control_variables': 5,
   'response_variables': 4,
   'model_specification': 4,
   'conclusions': 5,
   'overall_similarity': 5}, (1, 0): {'independent_variables': 4,
   'control_variables': 3,
   'response_variables': 4,
   'model_specification': 3,
   'conclusions': 1,
   'overall_similarity': 2}, (1, 1): {'independent_variables': 4,
   'control_variables': 4,
   'response_variables': 4,
   'model_specification': 3,
   'conclusions': 1,
   'overall_similarity': 2}, (1, 2): {'independent_variables': 3,
   'control_variables': 3,
   'response_variables': 4,
   'model_specification': 3,
   'conclusi

In [34]:
# get average similarity score for each subcategory within each group
average_within_group = {}
for group_id, comparisons in within_group.items():
    category_sums = {
        "independent_variables": 0,
        "control_variables": 0,
        "response_variables": 0,
        "model_specification": 0,
        "conclusions": 0,
        "overall_similarity": 0
    }
    num_comparisons = len(comparisons)
    
    for comparison, scores in comparisons.items():
        for category, score in scores.items():
            category_sums[category] += score
    
    average_scores = {category: total / num_comparisons
                      for category, total in category_sums.items()}
    average_within_group[group_id] = average_scores

In [35]:
# show average within group rounded to nearest tenth
average_within_group

{1: {'independent_variables': 4.333333333333333,
  'control_variables': 3.6666666666666665,
  'response_variables': 4.0,
  'model_specification': 4.0,
  'conclusions': 2.3333333333333335,
  'overall_similarity': 3.3333333333333335},
 2: {'independent_variables': 4.333333333333333,
  'control_variables': 4.0,
  'response_variables': 4.666666666666667,
  'model_specification': 3.3333333333333335,
  'conclusions': 4.666666666666667,
  'overall_similarity': 3.3333333333333335},
 3: {'independent_variables': 4.0,
  'control_variables': 3.6666666666666665,
  'response_variables': 4.666666666666667,
  'model_specification': 3.3333333333333335,
  'conclusions': 3.3333333333333335,
  'overall_similarity': 3.3333333333333335}}

In [36]:
# get average similarity score for each subcategory between each group
average_between_group = {}
for group_pair, comparisons in between_group.items():
    category_sums = {
        "independent_variables": 0,
        "control_variables": 0,
        "response_variables": 0,
        "model_specification": 0,
        "conclusions": 0,
        "overall_similarity": 0
    }
    num_comparisons = len(comparisons)
    
    for comparison, scores in comparisons.items():
        for category, score in scores.items():
            category_sums[category] += score
    
    average_scores = {category: total / num_comparisons
                      for category, total in category_sums.items()}
    average_between_group[group_pair] = average_scores

In [37]:
average_between_group

{(1, 2): {'independent_variables': 3.888888888888889,
  'control_variables': 3.6666666666666665,
  'response_variables': 3.7777777777777777,
  'model_specification': 3.111111111111111,
  'conclusions': 2.2222222222222223,
  'overall_similarity': 2.5555555555555554},
 (1, 3): {'independent_variables': 3.3333333333333335,
  'control_variables': 2.888888888888889,
  'response_variables': 2.6666666666666665,
  'model_specification': 2.5555555555555554,
  'conclusions': 2.888888888888889,
  'overall_similarity': 2.111111111111111},
 (2, 3): {'independent_variables': 3.888888888888889,
  'control_variables': 3.2222222222222223,
  'response_variables': 3.0,
  'model_specification': 3.0,
  'conclusions': 2.2222222222222223,
  'overall_similarity': 2.5555555555555554}}